<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 3 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Model Semantics, Partitioning, and Bucketing</div>
</div>

Write the same three rows into a Duplicate, a Unique, and an Aggregate Key table and compare what each keeps. Then partition the ten orders by day, bucket them by order ID, and read in three plans how a filter on the date and one on the order ID narrow what a query scans. Plan about 25 minutes.

**Module 3: Doris Table Models, Partitioning, and Bucketing** · [Lesson](course.md) · [Quiz](quiz03_models_and_data_distribution.ipynb) · [Course README](../../README.md)

## Lab scope

The Lab rebuilds only `orders_duplicate`, `orders_unique`, `orders_aggregate`, and `orders_partitioned`; the practices rebuild `orders_model_practice`. It needs the course sandbox from Lab 1.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()




## 1. Same keys, different models

Take order 1 (2300.00) and order 2 (405.00) from the WWI sample and simulate correcting order 1 to 2250.00 in the lab tables. After completing three submissions in order, observe:

- Duplicate Key retains all three input records.
- Unique Key retains two orders, with an amount of 2250.00 for order 1.
- Aggregate Key sums by key, producing 4550.00 for order 1.

The last result is suitable for additive metrics; to represent the current amount of an order after correction, use update-by-key semantics.

The Unique Key table sets `"enable_unique_key_merge_on_write" = "true"`: **Merge-on-Write** (MoW) resolves rows with the same key when they are written, so that a query reads only the latest one. It is the default for Unique Key tables in Doris 4.x; Module 7 shows what it does with versions and partial updates.

In [ ]:
definitions = {
    "orders_duplicate": 'CREATE TABLE orders_duplicate (id BIGINT, amount DECIMAL(12,2)) DUPLICATE KEY(id) DISTRIBUTED BY HASH(id) BUCKETS 1 PROPERTIES("replication_num"="1")',
    "orders_unique": 'CREATE TABLE orders_unique (id BIGINT, amount DECIMAL(12,2)) UNIQUE KEY(id) DISTRIBUTED BY HASH(id) BUCKETS 1 PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")',
    "orders_aggregate": 'CREATE TABLE orders_aggregate (id BIGINT, amount DECIMAL(12,2) SUM) AGGREGATE KEY(id) DISTRIBUTED BY HASH(id) BUCKETS 1 PROPERTIES("replication_num"="1")',
}
for table, ddl in definitions.items():
    lab.execute(f"DROP TABLE IF EXISTS {table}")
    show_sql("CREATE TABLE SQL", ddl)
    lab.execute(ddl)
    for row in [(1, "2300.00"), (1, "2250.00"), (2, "405.00")]:
        lab.insert(table, ["id", "amount"], [row])
expect(lab.query("SELECT id, amount FROM orders_duplicate ORDER BY id, amount"), [(1,"2250.00"),(1,"2300.00"),(2,"405.00")],
       title="Duplicate Key keeps all three rows")
expect(lab.query("SELECT id, amount FROM orders_unique ORDER BY id"), [(1,"2250.00"),(2,"405.00")],
       title="Unique Key keeps the later amount of order 1")
expect(lab.query("SELECT id, amount FROM orders_aggregate ORDER BY id"), [(1,"4550.00"),(2,"405.00")],
       title="Aggregate Key sums the two amounts of order 1")
for table in definitions:
    lab.sql(f"SELECT * FROM {table} ORDER BY id, amount", title=table + ": Results for the same input")

**Expected result**

Three table definitions and three green checks, then the three tables:

| Table | Rows |
| --- | --- |
| orders_duplicate | (1, 2250.00), (1, 2300.00), (2, 405.00) |
| orders_unique | (1, 2250.00), (2, 405.00) |
| orders_aggregate | (1, 4550.00), (2, 405.00) |

The same three writes give three answers: every row, the latest row per key, and a sum per key.

## 2. What partitioning and bucketing each control

The partitioned table uses the original ten orders: one partition per day, with each partition Hash-bucketed by order ID into four buckets. Compare plans for a full-table query, a date-only filter, and a combined date and order ID filter in order, and observe how the selected partition and Tablet ranges narrow.

This table retains historical detail records, with date and order ID used for sorting; the business unique key for a current-order table must be designed separately.


In [ ]:
from dw_course.wwi import sample
lab.execute("DROP TABLE IF EXISTS orders_partitioned")
lab.execute("""
CREATE TABLE orders_partitioned (
    order_date DATE, order_id BIGINT, amount DECIMAL(18,2)
) DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p_day1 VALUES [('2013-01-01'), ('2013-01-02')),
    PARTITION p_day2 VALUES [('2013-01-02'), ('2013-01-03'))
)
DISTRIBUTED BY HASH(order_id) BUCKETS 4
PROPERTIES("replication_num"="1")
""")
lab.insert("orders_partitioned", ["order_date", "order_id", "amount"],
           [(r["order_date"], r["order_id"], r["order_amount"]) for r in sample()["orders"]])
for query in [
    "SELECT * FROM orders_partitioned",
    "SELECT * FROM orders_partitioned WHERE order_date = '2013-01-01'",
    "SELECT * FROM orders_partitioned WHERE order_date = '2013-01-01' AND order_id = 1",
]:
    lab.sql("EXPLAIN " + query)
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM orders_partitioned WHERE order_date = '2013-01-01'"),
       [(5, "3944.20")], title="Pruning does not change the answer: five orders, 3944.20 on the first day")

**Expected result**

Three plans. In each, find the scan node `VOlapScanNode` and its lines `partitions=` and `tablets=`:

| Query | partitions= | tablets= |
| --- | --- | --- |
| No filter | `2/2 (p_day1,p_day2)` | `8/8` |
| `order_date = '2013-01-01'` | `1/2 (p_day1)` | `4/4` |
| `order_date = '2013-01-01' AND order_id = 1` | `1/2 (p_day1)` | `1/4` |

The date filter keeps one partition of two; then `tablets=` counts the tablets of the partitions that remain, 4. The order ID, the bucketing column, picks the one bucket whose tablet can hold order 1. The filters also appear as `PREDICATES`. A green check follows: the first day still has five orders and 3944.20. A plan shows what a query will read, not how long it takes.

## 3. Practice: which result can a report use?

A report needs the current amount of order 1 after its correction. Query order 1 in all three model tables, and the first day's orders and amount in `orders_partitioned`. Which table answers the report's question, and why is 4550.00 not an amount that order 1 ever had?

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
lab.sql("""SELECT 'duplicate' AS model, COUNT(*) AS rows_for_order_1, SUM(amount) AS summed FROM orders_duplicate WHERE id = 1
UNION ALL SELECT 'unique', COUNT(*), SUM(amount) FROM orders_unique WHERE id = 1
UNION ALL SELECT 'aggregate', COUNT(*), SUM(amount) FROM orders_aggregate WHERE id = 1
ORDER BY model""", title="Order 1 in the three models");
expect(lab.query("SELECT amount FROM orders_unique WHERE id = 1"), [("2250.00",)],
       title="Only Unique Key gives the corrected amount")
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM orders_partitioned WHERE order_date = '2013-01-01'"),
       [(5, "3944.20")], title="The first day in orders_partitioned: five orders, 3944.20")

**Expected result**

Three rows: `aggregate` with 1 row and 4550.00, `duplicate` with 2 rows that sum to 4550.00, and `unique` with 1 row of 2250.00, then two green checks. Only the Unique Key table holds the corrected amount, 2250.00. The Aggregate table added the original and the corrected amount, which is right for something additive, such as two sales, and wrong for two versions of one amount. The partitioned table still has the original 2300.00 for order 1: it holds the ten original orders, five of them on the first day, 3944.20.

## 4. Practice: choose a model for the current amount

The business needs to query the current amounts of orders. Receive (501,100), (501,80), and (502,40) in order, submitting each write only after the previous one completes. Choose a model for the separate table orders_model_practice, create it, and write the data; expect 80 for order 501 and 40 for order 502. Consider how the detail model and SUM aggregation would produce different results.

This exercise rebuilds only orders_model_practice.

Write your code in the next cell, then compare it with the reference solution after it.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_model_practice")
lab.execute("""CREATE TABLE orders_model_practice (order_id BIGINT, amount DECIMAL(12,2))
UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")""")
lab.execute("INSERT INTO orders_model_practice VALUES (501, 100)")
lab.execute("INSERT INTO orders_model_practice VALUES (501, 80)")
lab.execute("INSERT INTO orders_model_practice VALUES (502, 40)")
lab.sql("SELECT * FROM orders_model_practice ORDER BY order_id", title="Current amounts of the two orders");
expect(lab.query("SELECT * FROM orders_model_practice ORDER BY order_id"), [(501,"80.00"),(502,"40.00")],
       title="Order 501 is 80.00 and order 502 is 40.00")

**Expected result**

Two rows, 501 with 80.00 and 502 with 40.00, and a green check. A Unique Key table keeps the later amount of order 501; a Duplicate Key table would keep both rows of 501, and an Aggregate table with `SUM` would show 180.00.

## You are done when

- you can predict what a Duplicate, a Unique, and an Aggregate Key table keep from the same writes, and which one answers "what is the amount now";
- you can read `partitions=` and `tablets=` in a plan and say which filter narrowed which;
- the first day still has five orders and 3944.20 in every check: pruning changed what was read, not the answer.

## Next

Open [Quiz 3: Doris Table Models, Partitioning, and Bucketing](quiz03_models_and_data_distribution.ipynb), then continue to [Module 4: Querying External Data and Joining Lake Tables](../module04-external-access/course.md).
